> Notebook-friendly copy of `part-III/8.3-neural-weather-prediction-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

# 8.3) (Exercises) Neural Weather Prediction

![Ground truth, ensemble mean and ensemble standard deviation of a forecast over the Nordic region](_static/8.3-forecast-ground-truth-ensemble.png)

*Figure: A forecast result shown in the original exercise, which was built on the neural-lam code (the branch with the probabilistic model): the ground truth, the ensemble mean and the ensemble standard deviation over the Nordic region. The original exercise does not say which variable or lead time this is.*

**🎯 Learning objectives**

By the end of this notebook, you will be able to:

1. Describe how a limited-area weather forecast becomes a graph problem: grid nodes, mesh nodes, and the three kinds of edges between them,
2. Build a multiscale mesh over a grid and measure why it spreads information faster than a single-resolution mesh,
3. Write one message-passing layer in plain PyTorch and assemble the encode-process-decode forecast step from it, and
4. Roll a trained model forward autoregressively and read the forecast error against lead time.

**Graph neural networks** (GNNs) have become a practical tool for weather prediction because a forecast is a problem of spatial dependence: the temperature at one location tomorrow depends on the air that is upwind of it today. Representing the atmosphere as nodes and the interactions between them as edges lets one network learn those dependencies directly. The costs are real: weather models need high-resolution data and a lot of GPU time.

The model family this chapter builds on is **neural-lam**, a PyTorch code base for neural weather prediction on a *limited area* (a region with fixed lateral boundaries, rather than the whole globe). It implements a re-creation of GraphCast, a hierarchical model called Graph-FM, and the probabilistic Graph-EFM. Its example data come from MEPS, the operational ensemble forecast of the Nordic meteorological institutes, which runs at 2.5 km horizontal resolution.

Training those models takes GPUs, a pinned software stack from 2023, and several gigabytes of forecast data, so this notebook does not train them. It builds the same architecture at toy scale: a 24 × 24 grid, a three-level mesh, and two synthetic fields that drift with the wind. Every component has a counterpart in the real code, named where it appears, so that you can read the real code afterwards. The last section says how to run the real thing.

## 8.3.1 From a forecast to a graph

A numerical weather model advances a *state* $X_t$ (temperature, humidity, wind and so on at every grid cell) one time step at a time. The neural-lam models learn that step from data. To go from $X_{t-1}$ and $X_t$ to $X_{t+1}$, they pass information through three node sets:

1. **Grid nodes**, one per grid cell, which carry the state, plus *forcing* such as incoming solar radiation, and static fields.
2. **Mesh nodes**, a coarser set of nodes that does the actual processing. A mesh is cheaper than the grid, and it can be built with edges of several lengths, so that information travels far in few layers.
3. **Edges** of three kinds: grid to mesh (the *encoder*), mesh to mesh (the *processor*), and mesh to grid (the *decoder*).

The network predicts the *change* of the state, not the state itself, and adds it to $X_t$. Because the next step's input is this step's output, a forecast of $k$ steps is $k$ applications of the same network. neural-lam uses 3-hour steps, so its 19-step forecasts reach 57 hours. The code that does this is `predict_step` in `neural_lam/models/base_graph_model.py`.

Two mesh designs appear in the original exercise. In a **multiscale** mesh (GraphCast), all nodes live at the finest resolution, and the edges come in several lengths. In a **hierarchical** mesh (Graph-FM), each resolution is its own layer of nodes, and extra edges go up and down between layers. The next section builds both.

![Encode, process and decode steps of the hierarchical Hi-LAM model: grid nodes at the bottom, three mesh levels above](_static/8.3-hi-lam-encode-process-decode.png)

*Figure: The hierarchical model (Hi-LAM, called Graph-FM in the repository) as drawn in the original exercise. Left: the grid (red squares) and three mesh levels (blue circles). Right: the encoder moves information up the levels, the processor moves it down and up again, and the decoder brings it back to the grid.*

## 8.3.2 The grid and the mesh

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import shortest_path

# To make this notebook's output stable across runs
rnd_seed = 0
torch.manual_seed(rnd_seed)

Our "country" is the unit square. The forecast grid has 24 × 24 cells, and the mesh has three resolutions, 12 × 12, 6 × 6 and 3 × 3 nodes, listed from the finest (level 0) to the coarsest. Run the cell below to create them. The function `level_edges` connects every node of an $n \times n$ mesh to its four neighbors, in both directions, and returns the edges as two arrays: the sender ids and the receiver ids.

In [ ]:
# The forecast grid: 24 x 24 cell centers in the unit square
n_grid = 24
gx, gy = np.meshgrid((np.arange(n_grid) + 0.5) / n_grid, (np.arange(n_grid) + 0.5) / n_grid)
grid_xy = np.stack([gx.ravel(), gy.ravel()], axis=1)     # (576, 2)

In [ ]:
# Mesh nodes at three resolutions; level 0 is the finest
def mesh_level(n):
    x, y = np.meshgrid((np.arange(n) + 0.5) / n, (np.arange(n) + 0.5) / n)
    return np.stack([x.ravel(), y.ravel()], axis=1)

In [ ]:
mesh_sizes = [12, 6, 3]
levels = [mesh_level(n) for n in mesh_sizes]

In [ ]:
def level_edges(n, offset=0):
    """Directed edges between each node of an n x n mesh and its 4 neighbors; node ids are shifted by offset."""
    ids = np.arange(n * n).reshape(n, n)
    src = np.concatenate([ids[:, :-1].ravel(), ids[:, 1:].ravel(), ids[:-1, :].ravel(), ids[1:, :].ravel()])
    dst = np.concatenate([ids[:, 1:].ravel(), ids[:, :-1].ravel(), ids[1:, :].ravel(), ids[:-1, :].ravel()])
    return src + offset, dst + offset

In [ ]:
print([len(l) for l in levels], "mesh nodes per level")

## **Q1) Build the edges of the multiscale mesh**

In the multiscale mesh, only the 144 finest nodes exist. The edges of level $l$ connect finest nodes that are $2^l$ apart, in the same pattern as `level_edges` connects neighbors. Level 0 uses every node, level 1 uses every second node in each direction, and level 2 every fourth. Collect the edges of all three levels into one pair of arrays.

<details>
<summary><b>🔍 Hint</b></summary>

The finest nodes form a 12 × 12 array of ids, `ids = np.arange(144).reshape(12, 12)`. Taking every `k`-th row and column of it, `ids[k // 2::k, k // 2::k]`, gives the ids of the nodes that belong to the level with spacing `k`, centered in their blocks. `level_edges` returns edges between *positions* `0 ... m*m-1` of an `m` × `m` mesh, so look those positions up in the sub-array: `sub.ravel()[s]`.

</details>

In [ ]:
def multiscale_edges(n_fine, n_levels):
    ids = np.arange(n_fine * n_fine).reshape(n_fine, n_fine)
    S, D = [], []
    for l in range(n_levels):
        k = ___                       # spacing between the nodes of level l, in finest nodes
        sub = ___                     # ids of the finest nodes that belong to level l
        s, d = level_edges(sub.shape[0])
        S.append(___); D.append(___)  # translate positions in the sub-array into finest-node ids
    return np.concatenate(S), np.concatenate(D)

ms_src, ms_dst = multiscale_edges(mesh_sizes[0], len(mesh_sizes))
print(len(ms_src), "directed edges on", len(levels[0]), "nodes")

<details>
<summary><b>✅ A possible solution for Q1</b></summary>

```python
def multiscale_edges(n_fine, n_levels):
    ids = np.arange(n_fine * n_fine).reshape(n_fine, n_fine)
    S, D = [], []
    for l in range(n_levels):
        k = 2 ** l                                        # spacing between the nodes of level l
        sub = ids[k // 2::k, k // 2::k] if k > 1 else ids  # ids of the finest nodes in level l
        s, d = level_edges(sub.shape[0])
        S.append(sub.ravel()[s]); D.append(sub.ravel()[d])
    return np.concatenate(S), np.concatenate(D)

ms_src, ms_dst = multiscale_edges(mesh_sizes[0], len(mesh_sizes))
print(len(ms_src), "directed edges on", len(levels[0]), "nodes")
```

</details>

The hierarchical mesh keeps all three levels as separate nodes (144 + 36 + 9 = 189), connects neighbors within each level, and adds edges between each node and its nearest node one level up (and back down). Run the cell below to build it, then the one after to draw both meshes.

In [ ]:
# Hierarchical mesh: node ids are stacked level after level
offsets = np.cumsum([0] + [len(l) for l in levels])
h_src, h_dst = [], []
for l, n in enumerate(mesh_sizes):
    s, d = level_edges(n, offset=offsets[l])             # neighbors within level l
    h_src.append(s); h_dst.append(d)
for l in range(len(levels) - 1):
    _, nearest = cKDTree(levels[l + 1]).query(levels[l])  # nearest node one level up
    fine, coarse = np.arange(len(levels[l])) + offsets[l], nearest + offsets[l + 1]
    h_src += [fine, coarse]; h_dst += [coarse, fine]       # up and down edges
h_src, h_dst = np.concatenate(h_src), np.concatenate(h_dst)
h_xy = np.concatenate(levels)

In [ ]:
def plot_graph(xy, src, dst, ax, title, colors=None):
    segments = np.stack([xy[src], xy[dst]], axis=1)
    ax.add_collection(LineCollection(segments, colors=colors if colors is not None else "0.5", linewidths=0.6))
    ax.scatter(*xy.T, s=12, c="k", zorder=3)
    ax.set_title(title); ax.set_aspect("equal"); ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03); ax.axis("off")

fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
fine_src, fine_dst = level_edges(mesh_sizes[0])
plot_graph(levels[0], fine_src, fine_dst, ax[0], "Finest level only")
length = np.linalg.norm(levels[0][ms_src] - levels[0][ms_dst], axis=1)
plot_graph(levels[0], ms_src, ms_dst, ax[1], "Multiscale mesh", colors=plt.cm.viridis(np.round(np.log2(length / length.min())) / 2))
level_of = np.repeat(np.arange(3), [len(l) for l in levels])
h_colors = plt.cm.viridis(level_of[h_src] / 2)
h_colors[level_of[h_src] != level_of[h_dst]] = (0.85, 0.2, 0.2, 1)   # edges between levels in red
plot_graph(h_xy, h_src, h_dst, ax[2], "Hierarchical mesh (all levels overlaid)", colors=h_colors)
plt.tight_layout()

## **Q2) Count nodes and edges, and measure how far information has to travel**

A message-passing layer moves information along one edge per layer, so the **diameter** of the mesh (the largest number of edges on the shortest path between two nodes) is the number of layers a signal needs to cross the domain. Compute the number of nodes, the number of edges, and the diameter for the finest-only mesh, the multiscale mesh and the hierarchical mesh.

<details>
<summary><b>🔍 Hint</b></summary>

`scipy.sparse.csgraph.shortest_path(graph, directed=True, unweighted=True)` returns the matrix of shortest path lengths between all pairs of nodes. Build `graph` as a sparse adjacency matrix with `coo_matrix((np.ones(n_edges), (src, dst)), shape=(n_nodes, n_nodes))`. The diameter is the largest entry.

</details>

In [ ]:
def diameter(src, dst, n_nodes):
    A = coo_matrix(___, shape=(n_nodes, n_nodes))
    return int(shortest_path(___, directed=True, unweighted=True).___())

for name, (src, dst, n_nodes) in {"finest only": (fine_src, fine_dst, 144),
                                  "multiscale": (ms_src, ms_dst, 144),
                                  "hierarchical": (h_src, h_dst, offsets[-1])}.items():
    print(f"{name:13s} nodes: ___  edges: ___  diameter: ___")

<details>
<summary><b>✅ A possible solution for Q2</b></summary>

```python
def diameter(src, dst, n_nodes):
    A = coo_matrix((np.ones(len(src)), (src, dst)), shape=(n_nodes, n_nodes))
    return int(shortest_path(A, directed=True, unweighted=True).max())

for name, (src, dst, n_nodes) in {"finest only": (fine_src, fine_dst, 144),
                                  "multiscale": (ms_src, ms_dst, 144),
                                  "hierarchical": (h_src, h_dst, offsets[-1])}.items():
    print(f"{name:13s} nodes: {n_nodes:4d}  edges: {len(src):5d}  diameter: {diameter(src, dst, n_nodes)}")
```

</details>

You should find 528 edges and a diameter of 22 for the finest-only mesh, 672 edges and a diameter of 10 for the multiscale mesh, and 189 nodes, 1032 edges and a diameter of 8 for the hierarchical mesh. The multiscale mesh adds only 27% more edges than the finest-only mesh and more than halves the number of layers needed to cross the domain. This is why GraphCast-style models use it: a processor of 4 to 16 layers can still connect distant regions, such as a low-pressure system and the weather it will bring a day later.

Last, the edges between the grid and the mesh. In our toy, each grid node sends to its 4 nearest mesh nodes (the encoder), and receives from the same 4 (the decoder). These are the `g2m` and `m2g` edges of neural-lam, which come with their own edge features there (lengths and relative positions) that we skip here.

In [ ]:
_, nearest_mesh = cKDTree(levels[0]).query(grid_xy, k=4)   # (576, 4)
g2m_src = np.repeat(np.arange(len(grid_xy)), 4); g2m_dst = nearest_mesh.ravel()
m2g_src, m2g_dst = g2m_dst, g2m_src                          # the decoder reverses the encoder
print(len(g2m_src), "grid-to-mesh edges")

## 8.3.3 Toy weather

We need weather to forecast. Each sample is a sequence of 12 time steps of two fields on the grid (think temperature and humidity anomalies), made of three random waves that drift across the domain with a fixed wind, $(0.045, 0.02)$ domain lengths per step, which is about one grid cell per step. The model never sees the wind: it has to infer it from the difference between $X_t$ and $X_{t-1}$, exactly as neural-lam models do from their two input states.

In [ ]:
wind = np.array([0.045, 0.02])         # domain lengths per time step

def make_sequence(rng, n_steps):
    modes = [(rng.integers(1, 4), rng.integers(0, 4), rng.uniform(0, 2 * np.pi)) for _ in range(3)]
    amps = rng.normal(size=(2, 3))
    out = []
    for t in range(n_steps):
        x = grid_xy[:, 0] - wind[0] * t; y = grid_xy[:, 1] - wind[1] * t
        out.append(np.stack([sum(amps[v, m] * np.sin(2 * np.pi * (kx * x + ky * y) + ph)
                                 for m, (kx, ky, ph) in enumerate(modes)) for v in range(2)], axis=1))
    return np.stack(out).astype(np.float32)                  # (time, grid node, variable)

rng = np.random.default_rng(rnd_seed)
train_seq = torch.as_tensor(np.stack([make_sequence(rng, 12) for _ in range(64)]))
test_seq = torch.as_tensor(np.stack([make_sequence(rng, 12) for _ in range(8)]))
print(train_seq.shape, "(sample, time, grid node, variable)")

fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
for a, t in zip(ax, [0, 3, 6, 9]):
    a.imshow(test_seq[0, t, :, 0].reshape(n_grid, n_grid), origin="lower", cmap="RdBu_r", vmin=-3, vmax=3)
    a.set_title(f"t = {t}"); a.axis("off")

## 8.3.4 One message-passing layer

Every block of the network is built from one layer, the *interaction layer* (`InteractionNet` in `neural_lam/interaction_net.py`). For each edge, a small network turns the sender's and receiver's hidden vectors into a *message*. All messages that arrive at a node are summed, and a second small network updates the node from its own vector and that sum. The update is residual: the node keeps its old vector and adds the change.

## **Q3) Sum the messages at each receiver**

In the layer below, `msg` holds one message per edge, with shape `(batch, edges, d)`. Complete the aggregation so that `agg[:, i]` is the sum of the messages of all edges whose receiver is node `i`.

<details>
<summary><b>🔍 Hint</b></summary>

`tensor.index_add_(dim, index, source)` adds the slices of `source` into `tensor` at the positions given by `index` along `dim`, summing when an index repeats. The messages are stacked along dimension 1, and `self.dst` gives the receiver of each one.

</details>

In [ ]:
def mlp(sizes, layer_norm=True):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2: layers.append(nn.SiLU())
    if layer_norm: layers.append(nn.LayerNorm(sizes[-1]))
    return nn.Sequential(*layers)

In [ ]:
class InteractionLayer(nn.Module):
    """One round of message passing from sender nodes to receiver nodes, with a residual update of the receivers."""
    def __init__(self, src, dst, d):
        super().__init__()
        self.register_buffer("src", torch.as_tensor(src, dtype=torch.long))
        self.register_buffer("dst", torch.as_tensor(dst, dtype=torch.long))
        self.edge_mlp = mlp([2 * d, d, d])
        self.node_mlp = mlp([2 * d, d, d])

    def forward(self, h_send, h_recv):
        # One message per edge, from the vectors of its sender and receiver: (batch, edges, d)
        msg = self.edge_mlp(torch.cat([h_send[:, self.src], h_recv[:, self.dst]], dim=-1))
        agg = torch.zeros_like(h_recv).___(___, ___, ___)   # sum the messages arriving at each receiver
        return h_recv + self.node_mlp(torch.cat([h_recv, agg], dim=-1))

<details>
<summary><b>✅ A possible solution for Q3</b></summary>

```python
def mlp(sizes, layer_norm=True):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2: layers.append(nn.SiLU())
    if layer_norm: layers.append(nn.LayerNorm(sizes[-1]))
    return nn.Sequential(*layers)

class InteractionLayer(nn.Module):
    """One round of message passing from sender nodes to receiver nodes, with a residual update of the receivers."""
    def __init__(self, src, dst, d):
        super().__init__()
        self.register_buffer("src", torch.as_tensor(src, dtype=torch.long))
        self.register_buffer("dst", torch.as_tensor(dst, dtype=torch.long))
        self.edge_mlp = mlp([2 * d, d, d])
        self.node_mlp = mlp([2 * d, d, d])

    def forward(self, h_send, h_recv):
        # One message per edge, from the vectors of its sender and receiver: (batch, edges, d)
        msg = self.edge_mlp(torch.cat([h_send[:, self.src], h_recv[:, self.dst]], dim=-1))
        agg = torch.zeros_like(h_recv).index_add_(1, self.dst, msg)   # sum the messages arriving at each receiver
        return h_recv + self.node_mlp(torch.cat([h_recv, agg], dim=-1))
```

</details>

## 8.3.5 Encode, process, decode

The forecast step assembles interaction layers in the order of `predict_step` in neural-lam: embed the grid features, send them to the mesh (encoder), run several layers on the mesh (processor), send the result back to the grid (decoder), map to the output size, and add the predicted change to $X_t$. Two details come from the real code. The predicted change is first rescaled with the mean and standard deviation of one-step changes in the training data (`diff_mean`, `diff_std`; the script `create_parameter_weights.py` computes them), so that the network can output values near 1 whatever the variable's units. The input also carries the grid position, standing in for neural-lam's static grid features.

In [ ]:
class ToyNeuralLAM(nn.Module):
    def __init__(self, d=32, n_process=2, n_state=2):
        super().__init__()
        n_grid_in = 2 * n_state + 2                       # X_t, X_{t-1}, and the (x, y) position of the node
        self.grid_embed = mlp([n_grid_in, d, d])
        self.mesh_embed = mlp([2, d, d])                  # the mesh nodes only have a position
        self.encode = InteractionLayer(g2m_src, g2m_dst, d)
        self.process = nn.ModuleList([InteractionLayer(ms_src, ms_dst, d) for _ in range(n_process)])
        self.decode = InteractionLayer(m2g_src, m2g_dst, d)
        self.out = mlp([d, d, n_state], layer_norm=False)
        self.register_buffer("grid_xy", torch.as_tensor(grid_xy, dtype=torch.float32))
        self.register_buffer("mesh_xy", torch.as_tensor(levels[0], dtype=torch.float32))
        self.register_buffer("diff_std", torch.ones(n_state))
        self.register_buffer("diff_mean", torch.zeros(n_state))

    def forward(self, x_t, x_prev):
        B = x_t.shape[0]
        h_grid = self.grid_embed(torch.cat([x_t, x_prev, self.grid_xy.expand(B, -1, -1)], dim=-1))
        h_mesh = self.encode(h_grid, self.mesh_embed(self.mesh_xy).expand(B, -1, -1))   # grid -> mesh
        for layer in self.process:
            h_mesh = layer(h_mesh, h_mesh)                                              # mesh -> mesh
        h_grid = self.decode(h_mesh, h_grid)                                            # mesh -> grid
        delta = self.out(h_grid) * self.diff_std + self.diff_mean                       # predicted change, in physical units
        return x_t + delta                                                              # residual update of the state

In [ ]:
model = ToyNeuralLAM()
print(sum(p.numel() for p in model.parameters()), "trainable parameters")

Training is one-step prediction: pick a random sample and time $t$, feed $X_t$ and $X_{t-1}$, and compare the output with $X_{t+1}$. This takes under a minute on a laptop CPU. The real models are trained the same way, with `--ar_steps` controlling how many steps are unrolled inside the loss.

In [ ]:
# One-step change statistics, used to rescale the network output
diffs = (train_seq[:, 2:] - train_seq[:, 1:-1]).reshape(-1, 2)
model.diff_std.copy_(diffs.std(0)); model.diff_mean.copy_(diffs.mean(0))

n_steps = 3000
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_steps)
for step in range(n_steps):
    b = torch.randint(0, train_seq.shape[0], (16,))        # 16 random samples
    t = torch.randint(1, train_seq.shape[1] - 1, (16,))    # and a random time step in each
    x_prev, x_t, x_next = train_seq[b, t - 1], train_seq[b, t], train_seq[b, t + 1]
    loss = ((model(x_t, x_prev) - x_next) ** 2).mean()
    optimizer.zero_grad(); loss.backward(); optimizer.step(); scheduler.step()
    if step % 500 == 0:
        print(f"step {step:4d}  one-step loss {loss.item():.4f}")

## 8.3.6 Forecasting

A forecast starts from two known states, $X_0$ and $X_1$, and applies the model repeatedly, feeding each prediction back in. The model sees no truth after the start, apart from one thing the limited-area setting provides: the edge of the domain is not predicted but **prescribed**. In neural-lam, `unroll_prediction` overwrites a border strip of the grid with the true state (from a larger model) after every step. Our toy does the same with a strip two cells wide.

## **Q4) Roll the model forward for 9 steps**

Complete the loop so that `pred_steps[k]` is the forecast for step $k+2$. When `use_border` is true, replace the border cells of each prediction with the true state at that step before using it as input to the next step.

<details>
<summary><b>🔍 Hint</b></summary>

The conditioning states move forward after each step: the current state becomes the previous state, and the new state becomes the current one. The border mask has shape `(grid nodes, 1)` and equals 1 on the border, so `border * truth + (1 - border) * pred` keeps the truth on the border and the prediction elsewhere.

</details>

In [ ]:
# 1 on the two outermost rings of grid cells, 0 elsewhere
idx = np.arange(n_grid)
border = torch.as_tensor(((idx[:, None] < 2) | (idx[:, None] >= n_grid - 2) |
                          (idx[None, :] < 2) | (idx[None, :] >= n_grid - 2)).reshape(-1, 1), dtype=torch.float32)

In [ ]:
def rollout(model, seq, n_steps, use_border):
    """Forecast steps 2 ... n_steps + 1 of every sequence in seq, starting from steps 0 and 1."""
    state_prev, state = seq[:, 0], seq[:, 1]
    preds = []
    with torch.no_grad():
        for k in range(n_steps):
            pred = ___                                         # one step with the model
            if use_border:
                pred = ___                                     # prescribe the true state on the border
            preds.append(pred)
            state_prev, state = ___, ___                       # advance the conditioning states
    return torch.stack(preds, dim=1)                           # (sample, step, grid node, variable)

In [ ]:
forecast = rollout(model, test_seq, 9, use_border=True)
forecast_free = rollout(model, test_seq, 9, use_border=False)

<details>
<summary><b>✅ A possible solution for Q4</b></summary>

```python
# 1 on the two outermost rings of grid cells, 0 elsewhere
idx = np.arange(n_grid)
border = torch.as_tensor(((idx[:, None] < 2) | (idx[:, None] >= n_grid - 2) |
                          (idx[None, :] < 2) | (idx[None, :] >= n_grid - 2)).reshape(-1, 1), dtype=torch.float32)

def rollout(model, seq, n_steps, use_border):
    """Forecast steps 2 ... n_steps + 1 of every sequence in seq, starting from steps 0 and 1."""
    state_prev, state = seq[:, 0], seq[:, 1]
    preds = []
    with torch.no_grad():
        for k in range(n_steps):
            pred = model(state, state_prev)                              # one step with the model
            if use_border:
                pred = border * seq[:, k + 2] + (1 - border) * pred      # prescribe the true state on the border
            preds.append(pred)
            state_prev, state = state, pred                              # advance the conditioning states
    return torch.stack(preds, dim=1)                                     # (sample, step, grid node, variable)

forecast = rollout(model, test_seq, 9, use_border=True)
forecast_free = rollout(model, test_seq, 9, use_border=False)
```

</details>

## **Q5) Plot the forecast error against lead time**

Compute the root-mean-square error (RMSE) of the three forecasts over all grid nodes, variables and test samples, at each of the 9 lead times: the model with the prescribed border, the model without it, and the **persistence** forecast, which predicts that nothing changes from $X_1$. Plot them against the lead time in steps. A forecast is only useful where it beats persistence.

<details>
<summary><b>🔍 Hint</b></summary>

The truth for forecast step `k` is `test_seq[:, k + 2]`; stack them with `test_seq[:, 2:11]`. The errors then have shape `(sample, step, grid node, variable)`. Average the squares over every dimension except the step (`dim=(0, 2, 3)`) and take the square root. The persistence forecast is `test_seq[:, 1:2]`, which broadcasts over the steps.

</details>

In [ ]:
truth = test_seq[:, 2:11]                              # (sample, step, grid node, variable)

def rmse_by_lead_time(prediction):
    return ___

lead = np.arange(1, 10)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lead, ___, "o-", label="model, prescribed border")
ax.plot(lead, ___, "o-", label="model, free border")
ax.plot(lead, ___, "k--", label="persistence")
ax.set_xlabel("Lead time (steps)"); ax.set_ylabel("RMSE"); ax.legend();

<details>
<summary><b>✅ A possible solution for Q5</b></summary>

```python
truth = test_seq[:, 2:11]                              # (sample, step, grid node, variable)

def rmse_by_lead_time(prediction):
    return ((prediction - truth) ** 2).mean(dim=(0, 2, 3)).sqrt().numpy()

lead = np.arange(1, 10)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(lead, rmse_by_lead_time(forecast), "o-", label="model, prescribed border")
ax.plot(lead, rmse_by_lead_time(forecast_free), "o-", label="model, free border")
ax.plot(lead, rmse_by_lead_time(test_seq[:, 1:2].expand_as(truth)), "k--", label="persistence")
ax.set_xlabel("Lead time (steps)"); ax.set_ylabel("RMSE"); ax.legend();
```

</details>

Run once on a laptop CPU, the RMSE of the model with the prescribed border grew from about 0.10 at the first lead time to about 0.74 at the ninth, the free-border model from 0.13 to 1.08, and persistence went from 0.80 to 1.69 (the exact values shift slightly between machines). Three things are visible in these curves:

1. **Errors compound.** Each step starts from the previous prediction, so a small error at step 2 is part of the input at step 3. The error grows by a factor of 7 to 8 over nine steps even though the one-step loss during training was small. This is why neural-lam can train with `--ar_steps` greater than 1, which unrolls the model inside the loss so that it learns to cope with its own mistakes, and why forecast skill is always reported as a function of lead time.
2. **The boundary matters in a limited area.** The free-border model is worse at every lead time. The grid graph has no edges that wrap around, so the nodes at the edge have fewer neighbors than the others, and errors there spread inward. Prescribing the border also gives the model information it cannot have: what is entering the domain from outside.
3. **Persistence is a baseline, not a reference.** The persistence error is not monotonic because the toy waves are periodic, so the field drifts away from its starting state and then partly returns. The model stays below it at every lead time here, which is a low bar: the toy field translates rigidly, and real weather does not. This experiment says nothing about the skill of the real models. What it shows is the structure: how the graph, the interaction layers, the residual update and the rollout fit together.

## 8.3.7 Running the real models

The toy and the real models share the structure above and differ in scale. All numbers in the table come from the files in the course archive described below.

| | This notebook | neural-lam example data |
|---|---|---|
| Grid | 24 × 24 = 576 nodes | 268 × 238 = 63,784 nodes |
| Multiscale mesh | 144 nodes, 672 edges | 6,561 nodes, 57,616 edges |
| Hierarchical mesh | 144 + 36 + 9 nodes | 6,561 + 729 + 81 nodes |
| Edges within each level | 528 + 120 + 24 | 51,520 + 5,512 + 544 |
| Mesh to grid | 4 per grid node | 4 per grid node (255,136 edges) |
| Grid to mesh | 4 per grid node | 100,656 edges, about 15 per mesh node |
| Time step | one unit | 3 hours |
| Parameters | 29,282 | trained with 64-dimensional hidden vectors and 4 processor layers by default |

The coarsening factor between levels is 3 in the real hierarchical mesh and 2 in our toy, and the real graphs carry edge features (lengths and relative positions) that the toy does not use.

The original version of this exercise trained and evaluated two of the models, GraphCast and Graph-FM, on one of the example samples, using the neural-lam code on a GPU. The steps were, in order, from the neural-lam README:

```bash
# Static files: the graphs, the grid features, and the statistics of the data
python create_mesh.py --graph multiscale
python create_mesh.py --graph hierarchical --hierarchical 1 --levels 3
python create_grid_features.py
python create_parameter_weights.py

# Train, then evaluate on the validation and test sets
python train_model.py --model graphcast
python train_model.py --model graph_fm --graph hierarchical
python train_model.py --model graphcast --eval test
```

### What the original exercise produced

The figures below are from the original exercise, run by its authors with the neural-lam code on a GPU: a GraphCast model and a Graph-FM model, each with 4 processor layers of width 64, evaluated on the validation and test splits of the example data. They are not reproduced here, so treat them as illustrations of what the evaluation looks like, not as results you can check.

The first figure maps the loss at the longest lead time, 19 steps or 57 hours, for each model on each split. The error is not uniform: for Graph-FM it is largest over the Scandinavian mountains, where the terrain changes sharply, and for GraphCast on the test split it is largest in the east of the domain.

![Maps of the test loss at 57 hours lead time for Graph-FM and GraphCast, on the test and validation splits](_static/8.3-test-loss-maps-57h.png)

The next two figures give the RMSE of every variable at every lead time, in the units of the variable, for GraphCast and Graph-FM. As in the toy forecast of Q5, the error grows with lead time for almost every variable: the 2 m temperature (`t_2`) RMSE goes from 2.4 K at 3 hours to 7.3 K at 57 hours for GraphCast, and from 2.3 K to 8.1 K for Graph-FM. The two models differ little overall, and which one is better depends on the variable and the lead time: at 57 hours Graph-FM is lower for surface pressure (`pres_0g`, 1,953 Pa against 2,082 Pa) and higher for `t_2`.

![RMSE of each variable at each lead time for GraphCast](_static/8.3-rmse-graphcast.png)

![RMSE of each variable at each lead time for Graph-FM](_static/8.3-rmse-graph-fm.png)

### Running it yourself

Three things stand in the way of running this today. The code in the course archive was written for Python 3.10, PyTorch 2.0.1 and PyTorch Geometric 2.3.1 with CUDA 11.7 wheels, which is an environment you have to recreate on purpose; the repository's `main` branch ([mllam/neural-lam](https://github.com/mllam/neural-lam)) is under active development and may differ from the branch this exercise used. The archive does not include `create_mesh.py`, `create_grid_features.py` or `create_parameter_weights.py`, which the original exercise asked students to write. The data are large: the example samples alone are 3.7 GB, and the README says the example data are far too little to train a useful model and are meant to check that an installation works.

The archive is hosted on a personal SharePoint link, which its owner can revoke. As of 2026-10-09 it resolves to a file of 3,103,473,055 bytes (2.89 GiB) with this checksum, which a `pooch` call can pin:

```python
import pooch

archive = pooch.retrieve(
    url="https://unils-my.sharepoint.com/:u:/g/personal/haokun_liu_unil_ch/EVp6Oym5i_RElMf7jYz9VboB7ZdpyYKuTlBWLeVq9fyPTA?e=qMibgG&download=1",
    known_hash="sha256:ad511b41c3d705e6353fe79f5a4c1bae4092844e221ed7cdbff2fc2d0e7075df",
    fname="neural-lam-prob_model_lam.zip", path=pooch.os_cache("mlees"),
    processor=pooch.Unzip(),
)
```

It holds the code, the graphs, four pretrained checkpoints from the paper (`paper_checkpoints/`), the checkpoints of the earlier course runs, and 12 sample files for the training, validation and test splits of `meps_example`. This cell is not run by the notebook, because of the size.

## Resources

- [neural-lam](https://github.com/mllam/neural-lam) — the repository the original exercise used, by Joel Oskarsson and Tomas Landelius (MIT licence). The course archive contains the branch of the paper below.
- Oskarsson, Landelius, Deisenroth and Lindsten, [*Probabilistic Weather Forecasting with Hierarchical Graph Neural Networks*](https://arxiv.org/abs/2406.04759) — the Graph-EFM model and its hierarchical Graph-FM predecessor, as listed in the repository README.
- Oskarsson et al. (2023), [arXiv:2309.17370](https://arxiv.org/abs/2309.17370) — the hierarchical model (called Hi-LAM there) and the MEPS limited area, as cited in the repository README.
- Lam et al. (2023), [arXiv:2212.12794](https://arxiv.org/abs/2212.12794) — GraphCast, whose architecture neural-lam re-implements, as cited in the repository README.
- [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/) — the library neural-lam uses for message passing; the notebook above does the same with `index_add_`.